# 

In [1]:
import pandas as pd
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
#전처리
train = pd.read_table('train.tsv')
# 'item_description' 결측치 : 'No description yet'과 동일한 값으로 처리
train['item_description'] = train['item_description'].fillna("No description yet")
# 'brand_name' 결측치 처리 :
# 1. 기존 브랜드명이 기재된 컬럼에는 1, 누락된 곳에는 0
# 2. 결측치처리 : 'No Brand'
train["has_brand"] = train["brand_name"].notna().astype(int)
train["brand_name"] = train["brand_name"].fillna("No Brand")
# 'category_name' 결측치 처리 : 'No Category/No category/No category' - 기존 카테고리 분류 형식과 동일하게
train["category_name"] = train["category_name"].fillna("No category/No category/No category")
# 처리 후 전체 결측치 현황 한눈에 보기
print("=== 결측치 처리 후 데이터 결측치 현황 ===")
print(train.isna().sum())
train[['cat_1', 'cat_2', 'cat_3']] = train['category_name'].str.split('/', n=2, expand=True)
print(train['cat_1'].value_counts().head(10))  # 상위 카테고리 확인

=== 결측치 처리 후 데이터 결측치 현황 ===
train_id             0
name                 0
item_condition_id    0
category_name        0
brand_name           0
price                0
shipping             0
item_description     0
has_brand            0
dtype: int64
cat_1
Women                     664385
Beauty                    207828
Kids                      171689
Electronics               122690
Men                        93680
Home                       67871
Vintage & Collectibles     46530
Other                      45351
Handmade                   30842
Sports & Outdoors          25342
Name: count, dtype: int64


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
import pandas as pd
import numpy as np
import re

train_Women = train[train['cat_1']=='Women']

brand_map = {
    r"\bvictoria(?:'|’)?s?\s+secret\b": "vs",  
    r"\bcalvin\s+klein\b": "ck",
    r"\bmichael\s+kors\b": "mk",
    r"\bamerican\s+eagle\b": "ae",
    r"\bkate\s+spade\b":"ks",
    r"\bkendra\s+scott\b": "kc",
    r"\blouis\s+vuitton\b": "lv",
    r"\bbrandy\s+melville\b":"bm",
    r"\bvera\s+bradley\b":"vd",
    r"\btory\s+burch\b":"tb",
    r"\bnorth\s+face\b":"nf",
    r"\bforever\s+21\b":"f2"
    }


def normalize_brands(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    for pattern, replacement in brand_map.items():
        text = re.sub(pattern, replacement, text)
    return text

train_Women['name'] = train_Women['name'].apply(normalize_brands)

no_use_word = ['rm', 'description','size'
    ,'condition','great','good','excellent',
    'worn','used','new','like','firm',
    'price','shipping','freeship','bundle',
    'tags','tag','brandnew','free']

custom_stopwords = list(ENGLISH_STOP_WORDS.union(no_use_word))

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    stop_words=custom_stopwords
)
X_name = vectorizer.fit_transform(train_Women['name'].fillna(""))



#이건 벡터화된거 빈도 총합만 보는겁니다
feature_names = vectorizer.get_feature_names_out()

weights = np.array(X_name.sum(axis=0)).ravel()

tfidf_df = pd.DataFrame({
    "token": feature_names,
    "weight": weights
}).sort_values("weight", ascending=False)

print(tfidf_df.head(30))

/var/folders/3m/2p4scd0107qfx4clslwxs3r80000gn/T/ipykernel_31899/1417390439.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_Women['name'] = train_Women['name'].apply(normalize_brands)


           token        weight
46421         vs  15159.802786
32826       pink  14023.941242
25246    lularoe  12571.658379
46780    vs pink   9027.786516
23849   leggings   8994.947809
13378      dress   8026.793824
4442       black   7974.147929
29208       nike   6470.641642
38492      shirt   6303.183257
30015        nwt   6105.235868
38772     shorts   5642.973224
6696         bra   5622.590432
48273      women   5519.489359
20888      jeans   5085.654569
43086       tank   4863.422669
27672         mk   4314.824189
30958         os   4314.205276
43445         tc   4259.316934
25707  lululemon   4251.817080
3294         bag   4047.658253
20608     jacket   3785.923197
47616      white   3681.778574
40051      small   3664.796560
6304       boots   3649.708416
43848        tee   3376.114903
1665          ae   3372.595336
10230      coach   3368.939896
23123      large   3346.625256
19382       hold   3296.337223
19820     hoodie   3294.528389


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
import pandas as pd
import numpy as np
import re

train_Women = train[train['cat_1']=='Women']

brand_map1 = {
    r"\bbrand\s+new": "brandnew",
    r"\bfree\s+ship(ping)?\b" :"freeship",
    r"\bgray\b": "grey",
    r"\bbnwt\b": "nwt",
    r"\bquestions\b":"question",
    r"\bcomfy\b":"comfortable"
    }  

def normalize_brands(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    for pattern, replacement in brand_map1.items():
        text = re.sub(pattern, replacement, text)
    return text

train_Women['item_description'] = train_Women['item_description'].apply(normalize_brands)


no_use_word = ['rm', 'description','victoria','secret','size','vs'
    ,'condition','great','good','excellent','worn','used','new','like','firm','price','shipping','freeship','bundle',
    'tags','tag','brandnew','free','authentic','just', 'times','save','bought','items','box',
    'brand','material','light','long','twice','wear','washed','nwot','nwt','stains','flaws',
    'gently','wore','home','10','don','check','comes','little','100',
    'retail','inside','big','ask','lularoe','question'
    ]

custom_stopwords = list(ENGLISH_STOP_WORDS.union(no_use_word))

vectorizer = TfidfVectorizer(
    max_features=200000,
    ngram_range=(1,2),
    min_df=2,
    stop_words=custom_stopwords
)
X_desc = vectorizer.fit_transform(train_Women['item_description'].fillna("").astype(str))

feature_names1 = vectorizer.get_feature_names_out()

weights = np.array(X_desc.sum(axis=0)).ravel()

tfidf_df = pd.DataFrame({
    "token": feature_names1,
    "weight": weights
}).sort_values("weight", ascending=False)

print(tfidf_df.head(40))

/var/folders/3m/2p4scd0107qfx4clslwxs3r80000gn/T/ipykernel_31899/4231168135.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_Women['item_description'] = train_Women['item_description'].apply(normalize_brands)


              token        weight
161440        small  10918.477574
20627         black   8956.598706
106336       medium   8356.667707
129099         pink   7537.939653
91833         large   6718.864851
47604          cute   6695.235394
173803        super   5522.336124
39261         color   5496.413015
124666      perfect   5494.434460
192375        white   5169.692151
64647           fit   5020.240965
194750        women   4858.332418
94694      leggings   4750.995940
163074        smoke   4600.809425
23055          blue   4388.629382
74111          grey   4249.515676
197424           xs   4130.559373
55571         dress   4031.680327
65564          fits   3922.391640
15238           bag   3852.648540
41385   comfortable   3534.293788
173841   super cute   3498.382842
153788        shirt   3347.003753
71730          gold   3183.452465
17973     beautiful   3081.632376
164106         soft   3044.588312
196897           xl   2976.351219
26505           bra   2929.201689
157533       s